# Financial Friction Observatory

## Task 1: Data Exploration

### Objective
Understand the structure, size, completeness and
quality of the CFPB consumer complaint dataset.

### Data Sources
- 2023.csv
- 2024.csv
- 2025.csv

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import  Path

In [3]:
files = Path(r'C:\Users\Admin\OneDrive\Desktop\project\p3\data\raw').glob('*.csv')

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481


## 2. Dataset Structure

In this section, we examine:
- Number of rows and columns
- Column names
- Data types
- Non-null values
- Memory consumption

In [4]:
def details():
    print('-'*40)
    print('Number of Rows:', len(df))
    print('')
    print('Number of columns:', df.shape[1])
    print('')
    print('Name of Columns:','\n',df.columns)
    print('')
    print('Data types:','\n',df.dtypes)
    print('')
    print('DataFrame Information:')
    print( df.info(memory_usage="deep"))
    print('')
    print('Shape of DataFrame:', df.shape)
    print('')
    print('Memory Usage:')
    print(df.memory_usage(deep=True).sum() / (1024 ** 2), "MB")
    print('-'*40)

details()

----------------------------------------
Number of Rows: 186889

Number of columns: 15

Name of Columns: 
 Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Company public response', 'Company', 'State', 'ZIP code', 'Tags',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Complaint ID'],
      dtype='str')

Data types: 
 Date received                     str
Product                           str
Sub-product                       str
Issue                             str
Sub-issue                         str
Company public response           str
Company                           str
State                             str
ZIP code                          str
Tags                              str
Submitted via                     str
Date sent to company              str
Company response to consumer      str
Timely response?                  str
Complaint ID                    int64
dtype: object

Data

## 3. Missing Value Analysis

Investigate missing values in each column to understand
the completeness of the dataset.

In [5]:
def missing():
    print('-'*40)
    print(df.isna().sum())
    print()

    columns = [
    'Tags',
    'Company public response',
    'Sub-issue',
    'State'
]

    for col in columns:
        print('-'*50)
        print(f'columns: {col}')
        print('Count',df[col].isna().sum())
        print('percentage:',df[col].isna().sum()/len(df) * 100,'%')

    print('-'*40)
    
    

missing()

----------------------------------------
Date received                        0
Product                              0
Sub-product                          5
Issue                                5
Sub-issue                          122
Company public response          92984
Company                              0
State                             1654
ZIP code                             0
Tags                            155015
Submitted via                        0
Date sent to company                 0
Company response to consumer         0
Timely response?                     0
Complaint ID                         0
dtype: int64

--------------------------------------------------
columns: Tags
Count 155015
percentage: 82.94495663201151 %
--------------------------------------------------
columns: Company public response
Count 92984
percentage: 49.75359705493636 %
--------------------------------------------------
columns: Sub-issue
Count 122
percentage: 0.06527939044031485 %
--------

In [6]:
columns = [
    'Tags',
    'Company public response',
    'Company response to consumer',
    'Timely response?'
]

for col in columns:
    print(f"\n{'-'*50}")
    # print(f"Column: {col}")
    print('column:', df[col].value_counts(dropna=False))


--------------------------------------------------
column: Tags
NaN                              155015
Servicemember                     15781
Older American                    12726
Older American, Servicemember      3367
Name: count, dtype: int64

--------------------------------------------------
column: Company public response
NaN                                                                                                                        92984
Company has responded to the consumer and the CFPB and chooses not to provide a public response                            76886
Company believes it acted appropriately as authorized by contract or law                                                   16232
Company believes the complaint provided an opportunity to answer consumer's questions                                        319
Company believes complaint represents an opportunity for improvement to better serve consumers                               213
Company believes com

## 4. Response Outcome and Timeliness Analysis

Investigate the relationship between the company's response
outcome and whether the response was submitted on time.

In [7]:
#Comparing timely respnse and company response to customer
pd.crosstab(
    df['Company response to consumer'],
    df['Timely response?'],
    margins=True
)

Timely response?,No,Yes,All
Company response to consumer,,,
Closed with explanation,1026,150921,151947
Closed with monetary relief,181,23834,24015
Closed with non-monetary relief,49,10761,10810
Untimely response,117,0,117
All,1373,185516,186889


## 6. Duplicate detection
There are two different types of duplicates we need to investigate.

In [8]:
df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481


In [9]:
def duplicate():
    print('-'*40)
    print(df.duplicated().sum())
    print(df['Complaint ID'].nunique())
    print('-'*40)


duplicate()


----------------------------------------
0
186889
----------------------------------------


## 7. Date quality investigation

In [10]:
#Earliest and Latest date of complaint dates recieved and complaint sent to company date
columns = [
    'Date received',
    'Date sent to company'
]

for col in columns:
    temp_col = pd.to_datetime(df[col], errors='coerce', utc=True)

    print('-'*40)
    print(col)
    print(f'earliest complaint: {temp_col.min()}')
    print(f'latest complaint: {temp_col.max()}')
    print(temp_col.isna().sum())

    if temp_col.isna().sum() == 0:
        print('There are no invalid and unparseable dates','\n')
    else:
        print('Dates need to be checked','\n')
    print('-'*40)



----------------------------------------
Date received
earliest complaint: 2023-01-01 01:01:41+00:00
latest complaint: 2025-12-31 23:58:31+00:00
0
There are no invalid and unparseable dates 

----------------------------------------
----------------------------------------
Date sent to company
earliest complaint: 2023-01-01 01:12:26+00:00
latest complaint: 2026-08-18 10:45:46+00:00
0
There are no invalid and unparseable dates 

----------------------------------------


In [11]:
#Datetime report
date_received = pd.to_datetime(df['Date received'], errors='coerce', utc=True)
date_sent = pd.to_datetime(df['Date sent to company'], errors='coerce', utc=True)

df['processing_delay_days'] = (date_sent - date_received).dt.days


def date():
    print('-'*40)
    print(df['processing_delay_days'].describe(),'\n')

    print(f'Negative dates: {(df['processing_delay_days'] < 0).sum()}')
    print(f'Maximum delay: {df['processing_delay_days'].max()}')
    print(f'Averaeg delay: {df['processing_delay_days'].mean()}')
    print(f'Number of records took more than 7 days: {(df['processing_delay_days'] > 7).sum()} ')
    print(f'Number of records took more than 30 days: {(df['processing_delay_days'] > 30).sum()}')
    print('-'*40)


date()

----------------------------------------
count    186889.000000
mean          0.971218
std           5.738789
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         269.000000
Name: processing_delay_days, dtype: float64 

Negative dates: 0
Maximum delay: 269
Averaeg delay: 0.971218209739471
Number of records took more than 7 days: 5830 
Number of records took more than 30 days: 2067
----------------------------------------


In [12]:
temp_df = df[df['processing_delay_days'] > 30]
print(f'Earliest complaint recieved date with greater than 30 days delay: {temp_df['Date received'].min()}')

Earliest complaint recieved date with greater than 30 days delay: 2023-01-03T15:36:56.000Z


In [13]:
print(f'Latest complaint recieved date with greater than 30 days delay: {temp_df['Date received'].max()}')

Latest complaint recieved date with greater than 30 days delay: 2025-12-31T19:58:39.000Z


In [14]:
# Company complaint count with delay greater than 30 days
temp_df['Company'].value_counts().head()

Company
BANK OF AMERICA, NATIONAL ASSOCIATION    186
WELLS FARGO & COMPANY                    145
JPMORGAN CHASE & CO.                     135
CapitalJ Inc. dba Juno                    86
Chime Financial Inc                       82
Name: count, dtype: int64

In [15]:
print(f'percentage of all complaints took more than 30 days to be sent: {(temp_df['Complaint ID'].count() / df['Complaint ID'].count()) * 100}%')

percentage of all complaints took more than 30 days to be sent: 1.1060040986895965%


### company-wise delay rates

In [16]:
company_delay = ((df.groupby('Company')['Complaint ID'].count() / df['Complaint ID'].count()) * 100).sort_values(ascending=False)
company_delay


Company
WELLS FARGO & COMPANY                    13.325022
JPMORGAN CHASE & CO.                     11.829482
BANK OF AMERICA, NATIONAL ASSOCIATION     9.528116
NAVY FEDERAL CREDIT UNION                 7.887570
CAPITAL ONE FINANCIAL CORPORATION         6.327285
                                           ...    
Marcadis Singer PA                        0.000535
MUTUAL OF OMAHA MORTGAGE, LLC             0.000535
Williams Kastner & Gibbs PLLC             0.000535
Weltman, Weinberg & Reis Co., L.P.A.      0.000535
Waypoint Resource Group, LLC              0.000535
Name: Complaint ID, Length: 671, dtype: float64

In [17]:
delayed_by_company = temp_df.groupby('Company')['Complaint ID'].count().fillna(0)
total_by_company = df.groupby('Company')['Complaint ID'].count().fillna(0)
((delayed_by_company / total_by_company) * 100).sort_values(ascending=False)

Company
Acorns Grow Incorporated                100.0
Barker Financial LLC                    100.0
Business Financing Corporation          100.0
Arative Technologies LLC  (Bancoli)     100.0
College Guidance Resources  LLC         100.0
                                        ...  
WorldRemit Corp.                          NaN
Yamaha Motor Finance Corporation USA      NaN
Zap Solutions, Inc.                       NaN
eMoneyUSA Holdings, LLC                   NaN
eToro USA LLC                             NaN
Name: Complaint ID, Length: 671, dtype: float64

In [18]:
# Companoes with greater than 1000 complaint count
updated = total_by_company[total_by_company >= 1000]
((delayed_by_company / updated) * 100).sort_values(ascending=False).head(5)
updated

Company
ALLY FINANCIAL INC.                                     2021
BANK OF AMERICA, NATIONAL ASSOCIATION                  17807
BMO BANK NATIONAL ASSOCIATION                           2021
Block, Inc.                                             5332
CAPITAL ONE FINANCIAL CORPORATION                      11825
CITIBANK, N.A.                                          8769
CITIZENS FINANCIAL GROUP, INC.                          3048
Chime Financial Inc                                    10777
DISCOVER BANK                                           1074
Early Warning Services, LLC                             1382
FIFTH THIRD FINANCIAL CORPORATION                       1790
Fidelity National Information Services, Inc. (FNIS)     1194
GOLDMAN SACHS BANK USA                                  2170
HUNTINGTON NATIONAL BANK, THE                           1816
JPMORGAN CHASE & CO.                                   22108
KEYCORP                                                 1296
M&T BANK CORPORA

In [19]:
delay_for_eligible = delayed_by_company.reindex(
    updated.index,
    fill_value=0
)
delay_for_eligible

Company
ALLY FINANCIAL INC.                                     13
BANK OF AMERICA, NATIONAL ASSOCIATION                  186
BMO BANK NATIONAL ASSOCIATION                           26
Block, Inc.                                             49
CAPITAL ONE FINANCIAL CORPORATION                       43
CITIBANK, N.A.                                          66
CITIZENS FINANCIAL GROUP, INC.                          58
Chime Financial Inc                                     82
DISCOVER BANK                                            2
Early Warning Services, LLC                             11
FIFTH THIRD FINANCIAL CORPORATION                       33
Fidelity National Information Services, Inc. (FNIS)      5
GOLDMAN SACHS BANK USA                                  45
HUNTINGTON NATIONAL BANK, THE                           21
JPMORGAN CHASE & CO.                                   135
KEYCORP                                                 20
M&T BANK CORPORATION                            

In [20]:
company_report = pd.DataFrame({
    'Total complaints': updated,
    'Delayed complaints': delay_for_eligible
})
company_report

,Total complaints,Delayed complaints
Company,,
ALLY FINANCIAL INC.,2021,13
"BANK OF AMERICA, NATIONAL ASSOCIATION",17807,186
BMO BANK NATIONAL ASSOCIATION,2021,26
"Block, Inc.",5332,49
CAPITAL ONE FINANCIAL CORPORATION,11825,43
"CITIBANK, N.A.",8769,66
"CITIZENS FINANCIAL GROUP, INC.",3048,58
Chime Financial Inc,10777,82
DISCOVER BANK,1074,2


In [21]:
# Highest delay rate
company_report['Delay Rate'] = (company_report['Delayed complaints'] / company_report['Total complaints']) * 100
company_report.sort_values('Delay Rate', ascending=False).head(5)

,Total complaints,Delayed complaints,Delay Rate
Company,,,
ROBINHOOD MARKETS INC.,1163,25,2.149613
GOLDMAN SACHS BANK USA,2170,45,2.073733
"CITIZENS FINANCIAL GROUP, INC.",3048,58,1.902887
FIFTH THIRD FINANCIAL CORPORATION,1790,33,1.843575
"Paypal Holdings, Inc",2060,36,1.747573


In [22]:
# Highest delayed complaint count
company_report.sort_values(
    'Delayed complaints',
    ascending=False
).head(5)

,Total complaints,Delayed complaints,Delay Rate
Company,,,
"BANK OF AMERICA, NATIONAL ASSOCIATION",17807,186,1.044533
WELLS FARGO & COMPANY,24903,145,0.582259
JPMORGAN CHASE & CO.,22108,135,0.610639
Chime Financial Inc,10777,82,0.760880
"CITIBANK, N.A.",8769,66,0.752651


### Validate dataset's scope

In [23]:
df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,processing_delay_days
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481,0


#### product

In [24]:
df['Product'].unique()

<StringArray>
['Checking or savings account']
Length: 1, dtype: str

In [25]:
df['Product'].value_counts()

Product
Checking or savings account    186889
Name: count, dtype: int64

#### Sub-product

In [26]:
def sub_product():
    print('-'*40)
    print(df['Sub-product'].unique().tolist())
    print(df['Sub-product'].nunique())
    print(df['Sub-product'].value_counts(dropna=False))
    print(df['Sub-product'].isna().sum())
    print('-'*40)

sub_product()

----------------------------------------
['Savings account', 'Checking account', 'CD (Certificate of Deposit)', 'Other banking product or service', nan, 'Credit reporting']
5
Sub-product
Checking account                    153074
Other banking product or service     15187
Savings account                      14830
CD (Certificate of Deposit)           3792
NaN                                      5
Credit reporting                         1
Name: count, dtype: int64
5
----------------------------------------


#### examin the unexpected categories

In [27]:
# ther are four unexpected categories 
# since our intended scope was to checking and savings account 
# we will now examine those unexpected categories

unexpected_categories = [
    'Other banking product or service',
    'CD (Certificate of Deposit)',
    'Credit reporting'
]

categories = df[df['Sub-product'].isin(unexpected_categories) |
                df['Sub-product'].isna()]

categories['Issue'].value_counts(dropna=False)

Issue
Managing an account                                                                 11254
Closing an account                                                                   3139
Opening an account                                                                   1901
Problem with a lender or other company charging your account                         1852
Problem caused by your funds being low                                                583
Incorrect information on your report                                                   85
Improper use of your report                                                            58
Problem with a company's investigation into an existing problem                        58
Problem with fraud alerts or security freezes                                          23
Credit monitoring or identity theft protection services                                21
NaN                                                                                     5
Prob

In [28]:
issue_counts = categories.groupby('Sub-product', dropna=False)['Issue'].value_counts(dropna=False)

In [29]:
issue_counts.groupby(level=0, group_keys=False).head(5)

Sub-product                       Issue                                                          
CD (Certificate of Deposit)       Managing an account                                                2365
                                  Closing an account                                                 1012
                                  Opening an account                                                  394
                                  Problem with a company's investigation into an existing problem       6
                                  Incorrect information on your report                                  6
Credit reporting                  Improper use of your report                                           1
Other banking product or service  Managing an account                                                8889
                                  Closing an account                                                 2127
                                  Problem with a lende

In [30]:
df[df['Sub-product'].isna()]['Issue']

59523    NaN
62132    NaN
62405    NaN
62872    NaN
67968    NaN
Name: Issue, dtype: str

#### Issue Analysis

In [49]:
# Now we will analyze issue

def issue():
    print('-'*40)
    print(f'All unique issue categories: {df['Issue'].unique()}')
    print(df['Issue'].value_counts().head(10))
    print(((df['Issue'].value_counts().head(10) / df['Complaint ID'].count()) * 100).round(2))
    print(f'missing issue values: {df['Issue'].isna().sum()}')
    print('-'*40)

issue()

----------------------------------------
All unique issue categories: <StringArray>
[                    'Problem with a lender or other company charging your account',
                                                              'Managing an account',
                                                               'Opening an account',
                                           'Problem caused by your funds being low',
                                                               'Closing an account',
                                             'Incorrect information on your report',
                          'Credit monitoring or identity theft protection services',
 'Problem with a credit reporting company's investigation into an existing problem',
                                                      'Improper use of your report',
                                    'Problem with fraud alerts or security freezes',
                                 'Unable to get your credit report

In [38]:
df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,processing_delay_days
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481,0


In [52]:
def sub_issue():
    print('-'*40)
    print(f'All unique sub-issue categories: {df['Sub-issue'].unique()}')
    print(df['Sub-issue'].value_counts().head(10))
    print(((df['Sub-issue'].value_counts().head(10) / df['Complaint ID'].count()) * 100).round(2))
    print(f'missing issue values: {df['Sub-issue'].isna().sum()}')
    print('-'*40)

sub_issue()

----------------------------------------
All unique sub-issue categories: <StringArray>
[                                                       'Transaction was not authorized',
            'Money was taken from your account on the wrong day or for the wrong amount',
                                                     'Problem using a debit or ATM card',
                                                  'Problem making or receiving payments',
                                                                        'Banking errors',
                                                             'Unable to open an account',
                                              'Non-sufficient funds and associated fees',
                                          'Funds not handled or disbursed as instructed',
                                                               'Deposits or withdrawals',
                                                                           'Fee problem',
            

In [51]:
pd.DataFrame(df.groupby('Issue')['Sub-issue'].value_counts())

count
Issue                                              Sub-issue                                                
Closing an account                                 Company closed your account                         11170
                                                   Funds not received from closed account               8511
                                                   Can't close your account                             4003
                                                   Fees charged for closing account                      692
Credit monitoring or identity theft protection ... Problem canceling credit monitoring or identify...     29
                                                   Billing dispute for services                           27
                                                   Didn't receive services that were advertised           19
                                                   Received unwanted marketing or advertising              8
                                                   Problem with product or service terms changing          7
Improper use of your report                        Reporting company used your report improperly         115
                                                   Credit inquiries on your report that you don't ...     44
                                                   Received unsolicited financial product or insur...     10
                                                   Report provided to employer without your writte...      3
Incorrect information on your report               Information belongs to someone else                   204
                                                   Account status incorrect                              109
                                                   Account information incorrect                          89
                                                   Personal information incorrect                         31
                                                   Old information reappears or never goes away           21
                                                   Public record information inaccurate                   14
                                                   Information is missing that should be on the re...     13
Managing an account                                Deposits and withdrawals                            38484
                                                   Problem using a debit or ATM card                   21934
                                                   Funds not handled or disbursed as instructed        12060
                                                   Banking errors                                      10224
                                                   Problem accessing account                            8030
                                                   Problem making or receiving payments                 5915
                                                   Fee problem                                          3041
                                                   Cashing a check                                      2907
                                                   Deposits or withdrawals                               733
                                                   Problem with renewal                                  244
                                                   Problem with fees or penalties                        177
Opening an account                                 Unable to open an account                            5574
                                                   Account opened without my consent or knowledge       5083
                                                   Didn't receive terms that were advertised            3729
                                                   Account opened as a result of fraud                  1600
                                                   Confusing or missin

In [56]:
# Understandig the relationship between issue and Sub issue

df[df['Issue'] == 'Managing an account']['Sub-issue'].value_counts().head(10)

Sub-issue
Deposits and withdrawals                        38484
Problem using a debit or ATM card               21934
Funds not handled or disbursed as instructed    12060
Banking errors                                  10224
Problem accessing account                        8030
Problem making or receiving payments             5915
Fee problem                                      3041
Cashing a check                                  2907
Deposits or withdrawals                           733
Problem with renewal                              244
Name: count, dtype: int64

In [57]:
df[df['Issue'] == 'Closing an account']['Sub-issue'].value_counts().head(10)

Sub-issue
Company closed your account               11170
Funds not received from closed account     8511
Can't close your account                   4003
Fees charged for closing account            692
Name: count, dtype: int64

#### Company analysis

In [66]:
def company():
    print('-'*40)
    print(f'number of distinct companies: {df['Company'].nunique()}')
    print(f'Top 10 companies by complaints: {df['Company'].value_counts().head(10)}')
    print(f'Percentage of total complaints represented by each: {((df['Company'].value_counts().head(10) / df['Complaint ID'].count()) * 100).round(2)}')
    print(f'Missing company values: {df['Company'].isna().sum()}')

company()

----------------------------------------
number of distinct companies: 671
Top 10 companies by complaints: Company
WELLS FARGO & COMPANY                    24903
JPMORGAN CHASE & CO.                     22108
BANK OF AMERICA, NATIONAL ASSOCIATION    17807
NAVY FEDERAL CREDIT UNION                14741
CAPITAL ONE FINANCIAL CORPORATION        11825
Chime Financial Inc                      10777
CITIBANK, N.A.                            8769
Block, Inc.                               5332
U.S. BANCORP                              4919
TRUIST FINANCIAL CORPORATION              4851
Name: count, dtype: int64
Percentage of total complaints represented by each: Company
WELLS FARGO & COMPANY                    13.33
JPMORGAN CHASE & CO.                     11.83
BANK OF AMERICA, NATIONAL ASSOCIATION     9.53
NAVY FEDERAL CREDIT UNION                 7.89
CAPITAL ONE FINANCIAL CORPORATION         6.33
Chime Financial Inc                       5.77
CITIBANK, N.A.                            4.69


#### Submission channel analysis

In [67]:
df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,processing_delay_days
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481,0


In [80]:
def submission_channel():
    print('-'*40)
    print(f'Distinct submission channels: {df['Submitted via'].unique().tolist()}')
    print(f'Compliant count per channles: {df['Submitted via'].value_counts()}')
    print(f'Percentage of total complaints per channel: {((df['Submitted via'].value_counts() / df['Complaint ID'].count()) * 100).round(2)}')
    print(f'Missing values: {df['Submitted via'].isna().sum()}')

submission_channel()

----------------------------------------
Distinct submission channels: ['Web', 'Referral', 'Phone', 'Postal mail']
Compliant count per channles: Submitted via
Web            161766
Referral        11894
Phone           11749
Postal mail      1480
Name: count, dtype: int64
Percentage of total complaints per channel: Submitted via
Web            86.56
Referral        6.36
Phone           6.29
Postal mail     0.79
Name: count, dtype: float64
Missing values: 0


#### Geographic  analysis

In [83]:
df.head(1)

,Date received,Product,Sub-product,Issue,Sub-issue,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID,processing_delay_days
0,2023-03-11T14:52:49.000Z,Checking or savings account,Savings account,Problem with a lender or other company chargin...,Transaction was not authorized,Company has responded to the consumer and the ...,"CITIBANK, N.A.",NY,11367,Older American,Web,2023-03-11T15:09:00.000Z,Closed with explanation,Yes,6681481,0


In [92]:
def georgraphy():
    print('-'*40)
    print(f'Number of distinct states: {df['State'].nunique()}')
    print(f'Top 10 States by complaint count: \n{df['State'].value_counts().head(10)}')
    print(f'Percentage of total complaints represented by each state: \n{((df['State'].value_counts().head(10) / df['Complaint ID'].count()) * 100).round(2)}')
    print(f'Missing state values: {df['State'].isna().sum()}')


georgraphy()

----------------------------------------
Number of distinct states: 61
Top 10 States by complaint count: 
State
CA    26326
FL    17898
TX    17212
NY    14654
GA    10118
IL     7536
PA     6825
NJ     6559
NC     6014
VA     5457
Name: count, dtype: int64
Percentage of total complaints represented by each state: 
State
CA    14.09
FL     9.58
TX     9.21
NY     7.84
GA     5.41
IL     4.03
PA     3.65
NJ     3.51
NC     3.22
VA     2.92
Name: count, dtype: float64
Missing state values: 1654


In [96]:
sorted(df['State'].dropna().unique())

['AA',
 'AE',
 'AK',
 'AL',
 'AP',
 'AR',
 'AS',
 'AZ',
 'CA',
 'CO',
 'CT',
 'DC',
 'DE',
 'FL',
 'GA',
 'GU',
 'HI',
 'IA',
 'ID',
 'IL',
 'IN',
 'KS',
 'KY',
 'LA',
 'MA',
 'MD',
 'ME',
 'MI',
 'MN',
 'MO',
 'MP',
 'MS',
 'MT',
 'NC',
 'ND',
 'NE',
 'NH',
 'NJ',
 'NM',
 'NV',
 'NY',
 'OH',
 'OK',
 'OR',
 'PA',
 'PR',
 'PW',
 'RI',
 'SC',
 'SD',
 'TN',
 'TX',
 'UNITED STATES MINOR OUTLYING ISLANDS',
 'UT',
 'VA',
 'VI',
 'VT',
 'WA',
 'WI',
 'WV',
 'WY']

In [104]:
postal_code = [
    'AA',
    'AE',
    'AS',
    'GU',
    'MP',
    'PR',
    'VI',
    'PW',
    'UNITED STATES MINOR OUTLYING ISLANDS',
    'AP',
    'DC'
    ]

In [105]:
df[df['State'].isin(postal_code)]['State'].value_counts()

State
DC                                      1093
PR                                       248
AE                                        72
AP                                        54
VI                                        25
GU                                        20
UNITED STATES MINOR OUTLYING ISLANDS      10
MP                                         4
AS                                         2
AA                                         1
PW                                         1
Name: count, dtype: int64